# Keyword Labels vs. Structured Benefit Categories

This notebook compares our keyword-based benefit labels (from `label_benefits.py` / `label_benefits_remote.py`) against the structured `BENEFIT_NAME`, `BENEFIT_SUBCATEGORY_NAME`, and `BENEFIT_CATEGORIES_NAME` columns in the original data.

**Goal:** Assess how well the keyword labels align with the structured fields — identify where they agree, where they diverge, and what each approach captures that the other misses.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

base = Path("..") / "data" / "processed"
data = pd.read_parquet(base / "labeled_v1.parquet")
print(f"Rows: {data.shape[0]:,}")

# Keyword labels
kw_labels = ["EDU_ASSISTANCE", "PAID LEAVE", "HEALTH_WELLBEING", "PARENTAL_LEAVE", "CULTURE", "REMOTE_KW", "WELLBEING"]

# Structured columns are stored as JSON-style string arrays — parse them
for col in ["BENEFIT_NAME", "BENEFIT_SUBCATEGORY_NAME", "BENEFIT_CATEGORIES_NAME"]:
    data[col] = data[col].apply(lambda x: json.loads(x) if isinstance(x, str) and x.startswith("[") else [])

print(f"Rows with any structured benefit: {data['BENEFIT_NAME'].apply(len).gt(0).sum():,}")
print(f"Rows with any keyword label: {data[kw_labels].any(axis=1).sum():,}")

Rows: 99,860
Rows with any structured benefit: 49,359
Rows with any keyword label: 41,703


## 1. Structured benefit field coverage

First, let's see what unique values appear in each structured column and how often.

In [6]:
# Explode each structured column to get individual values
for col in ["BENEFIT_CATEGORIES_NAME", "BENEFIT_SUBCATEGORY_NAME", "BENEFIT_NAME"]:
    exploded = data[col].explode().dropna()
    exploded = exploded[exploded != ""]
    print(f"\n=== {col} — {exploded.nunique()} unique values ===")
    display(exploded.value_counts().to_frame("count"))


=== BENEFIT_CATEGORIES_NAME — 8 unique values ===


,count
BENEFIT_CATEGORIES_NAME,
Insurance,83358
Paid Leave,36375
Retirement and Savings,35605
Education and Career Development,17070
Work-Life Balance,12983
Supplemental Pay,11856
Other Benefits,8700
Health and Wellness Benefits,8691



=== BENEFIT_SUBCATEGORY_NAME — 44 unique values ===


,count
BENEFIT_SUBCATEGORY_NAME,
401(k) Plans,26571
Paid Time Off (PTO),25902
Dental Insurance,21923
Vision Insurance,21117
Health Insurance,19117
Life Insurance,14336
Flexible Work Schedules,9816
Financial Aid/Assistance,8457
Other Retirement and Savings,8253



=== BENEFIT_NAME — 44 unique values ===


,count
BENEFIT_NAME,
401(k) Plans,26571
Paid Time Off (PTO),25902
Dental Insurance,21923
Vision Insurance,21117
Health Insurance,19117
Life Insurance,14336
Flexible Work Schedules,9816
Financial Aid/Assistance,8457
Other Retirement and Savings,8253


In [7]:
# Mapping: keyword label -> structured BENEFIT_SUBCATEGORY_NAME values
# CULTURE has no structured equivalent
keyword_to_structured = {
    "HEALTH_WELLBEING": ["Health and Wellness Programs", "Mental Health", "Bereavement/Mental Health Leave", "Health and Wellness Stipends","Health and Wellness Applications"],
    "PARENTAL_LEAVE": ["Parental Leave"],
    "PAID LEAVE": ["Sick Leave", "Floating Holidays", "Paid Time Off (PTO)"],
    "EDU_ASSISTANCE": ["Financial Aid/Assistance"],
}

# Build structured indicators from the benefit subcategory mapping
def has_any(series, terms):
    """Check if any of `terms` appear in the list-valued column."""
    terms_lower = [t.lower() for t in terms]
    return series.apply(lambda lst: any(v.lower() in terms_lower for v in lst if isinstance(v, str)))

for kw_col, struct_vals in keyword_to_structured.items():
    col_name = f"S_{kw_col}"
    data[col_name] = has_any(data["BENEFIT_SUBCATEGORY_NAME"], struct_vals)
    print(f"{col_name}: {data[col_name].sum():,} rows  (from {struct_vals})")

# REMOTE_KW maps to REMOTE_TYPE_NAME (separate structured column, not benefit subcategory)
data["REMOTE_STRUCTURED"] = data["REMOTE_TYPE_NAME"].isin(["Remote", "Hybrid Remote"])
print(f"REMOTE_STRUCTURED: {data['REMOTE_STRUCTURED'].sum():,} rows  (from REMOTE_TYPE_NAME = Remote or Hybrid Remote)")
print(f"\nREMOTE_TYPE_NAME distribution:")
print(data["REMOTE_TYPE_NAME"].value_counts(dropna=False))

S_HEALTH_WELLBEING: 8,598 rows  (from ['Health and Wellness Programs', 'Mental Health', 'Bereavement/Mental Health Leave', 'Health and Wellness Stipends', 'Health and Wellness Applications'])
S_PARENTAL_LEAVE: 4,490 rows  (from ['Parental Leave'])
S_PAID LEAVE: 27,275 rows  (from ['Sick Leave', 'Floating Holidays', 'Paid Time Off (PTO)'])
S_EDU_ASSISTANCE: 8,457 rows  (from ['Financial Aid/Assistance'])
REMOTE_STRUCTURED: 6,472 rows  (from REMOTE_TYPE_NAME = Remote or Hybrid Remote)

REMOTE_TYPE_NAME distribution:
REMOTE_TYPE_NAME
[None]           85146
Not Remote        8242
Remote            4939
Hybrid Remote     1533
Name: count, dtype: int64


## 2. Which structured benefit values co-occur with each keyword label?

Rather than pre-assuming mappings, let the data show which structured categories, subcategories, and benefit names appear most often among rows flagged by each keyword label.

In [8]:
kw_to_label = {
    "EDU_ASSISTANCE": "Tuition Assistance",
    "PAID LEAVE": "Paid Leave",
    "HEALTH_WELLBEING": "Health & Wellbeing",
    "PARENTAL_LEAVE": "Parental Leave",
    "CULTURE": "Inclusive Workplace",
    "REMOTE_KW": "Remote Work",
}

for kw_col, label in kw_to_label.items():
    flagged = data[data[kw_col].astype(bool)]
    print(f"\n{'='*70}")
    print(f"{label} ({kw_col}) — {len(flagged):,} rows flagged by keyword")
    print(f"{'='*70}")

    for struct_col in ["BENEFIT_CATEGORIES_NAME", "BENEFIT_SUBCATEGORY_NAME", "BENEFIT_NAME"]:
        exploded = flagged[struct_col].explode().dropna()
        exploded = exploded[exploded != ""]
        if len(exploded) == 0:
            print(f"\n  {struct_col}: no values")
            continue
        top = exploded.value_counts().head(10)
        total_flagged = len(flagged)
        print(f"\n  {struct_col} (top 10):")
        for val, count in top.items():
            print(f"    {val:45s}  {count:>6,}  ({count/total_flagged*100:5.1f}%)")



Tuition Assistance (EDU_ASSISTANCE) — 9,273 rows flagged by keyword

  BENEFIT_CATEGORIES_NAME (top 10):
    Insurance                                      20,427  (220.3%)
    Education and Career Development               10,724  (115.6%)
    Paid Leave                                     10,007  (107.9%)
    Retirement and Savings                          9,251  ( 99.8%)
    Supplemental Pay                                3,093  ( 33.4%)
    Health and Wellness Benefits                    2,786  ( 30.0%)
    Other Benefits                                  2,459  ( 26.5%)
    Work-Life Balance                               2,030  ( 21.9%)

  BENEFIT_SUBCATEGORY_NAME (top 10):
    Financial Aid/Assistance                        8,286  ( 89.4%)
    401(k) Plans                                    6,661  ( 71.8%)
    Paid Time Off (PTO)                             6,495  ( 70.0%)
    Vision Insurance                                5,300  ( 57.2%)
    Dental Insurance                    

## 3. Cross-tabulation: Keyword vs. Structured

For each mapped benefit, compare the keyword label against the structured indicator. CULTURE has no structured counterpart. REMOTE_KW is compared against `REMOTE_TYPE_NAME` (Remote/Hybrid Remote).

In [9]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# All keyword -> structured pairs (including REMOTE_KW via REMOTE_TYPE_NAME)
all_pairs = {
    "HEALTH_WELLBEING": ("S_HEALTH_WELLBEING", "Health & Wellbeing"),
    "PARENTAL_LEAVE": ("S_PARENTAL_LEAVE", "Parental Leave"),
    "PAID LEAVE": ("S_PAID LEAVE", "Paid Leave"),
    "EDU_ASSISTANCE": ("S_EDU_ASSISTANCE", "Tuition Assistance"),
    "REMOTE_KW": ("REMOTE_STRUCTURED", "Remote Work"),
}

summary_rows = []
for kw_col, (struct_col, name) in all_pairs.items():
    kw = data[kw_col].astype(bool)
    st = data[struct_col].astype(bool)

    tn, fp, fn, tp = confusion_matrix(st, kw).ravel()
    prec = precision_score(st, kw, zero_division=0)
    rec  = recall_score(st, kw, zero_division=0)
    f1   = f1_score(st, kw, zero_division=0)

    summary_rows.append({
        "Benefit": name,
        "Keyword+": int(kw.sum()),
        "Structured+": int(st.sum()),
        "Both+": int(tp),
        "KW only": int(fp),
        "Struct only": int(fn),
        "Precision": round(prec, 3),
        "Recall": round(rec, 3),
        "F1": round(f1, 3),
    })

summary = pd.DataFrame(summary_rows)
display(summary)

,Benefit,Keyword+,Structured+,Both+,KW only,Struct only,Precision,Recall,F1
0,Health & Wellbeing,4948,8598,3024,1924,5574,0.611,0.352,0.446
1,Parental Leave,4636,4490,4329,307,161,0.934,0.964,0.949
2,Paid Leave,28220,27275,26243,1977,1032,0.930,0.962,0.946
3,Tuition Assistance,9273,8457,8286,987,171,0.894,0.980,0.935
4,Remote Work,6359,6472,4133,2226,2339,0.650,0.639,0.644


## 4. Detailed cross-tabs per benefit

In [34]:
for kw_col, (struct_col, name) in all_pairs.items():
    kw = data[kw_col].astype(bool)
    st = data[struct_col].astype(bool)

    ct = pd.crosstab(st, kw, rownames=[f"Structured ({struct_col})"], colnames=[f"Keyword ({kw_col})"])
    struct_source = keyword_to_structured.get(kw_col, "REMOTE_TYPE_NAME (Remote/Hybrid)")
    print(f"\n{'='*60}")
    print(f"{name}  —  Structured = {struct_source}")
    print(f"{'='*60}")
    display(ct)


Health & Wellbeing  —  Structured = ['Health and Wellness Programs']


Keyword (HEALTH_WELLBEING),False,True
Structured (S_HEALTH_WELLBEING),,
False,89573,2067
True,5339,2881



Parental Leave  —  Structured = ['Parental Leave']


Keyword (PARENTAL_LEAVE),False,True
Structured (S_PARENTAL_LEAVE),,
False,95063,307
True,161,4329



Paid Leave  —  Structured = ['Sick Leave', 'Floating Holidays', 'Paid Time Off (PTO)']


Keyword (PAID LEAVE),False,True
Structured (S_PAID LEAVE),,
False,70608,1977
True,1032,26243



Tuition Assistance  —  Structured = ['Financial Aid/Assistance']


Keyword (EDU_ASSISTANCE),False,True
Structured (S_EDU_ASSISTANCE),,
False,90416,987
True,171,8286



Remote Work  —  Structured = REMOTE_TYPE_NAME (Remote/Hybrid)


Keyword (REMOTE_KW),False,True
Structured (S_REMOTE_KW),,
False,91162,2226
True,2339,4133


## 5. Inspect disagreements

For each benefit, sample rows where the keyword and structured labels disagree to understand *why* they diverge.

In [35]:
sample_n = 5

for kw_col, (struct_col, name) in all_pairs.items():
    kw = data[kw_col].astype(bool)
    st = data[struct_col].astype(bool)

    kw_only = data[kw & ~st]
    st_only = data[~kw & st]

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    if len(kw_only) > 0:
        print(f"\n--- Keyword+ / Structured- ({len(kw_only):,} rows) — sample {min(sample_n, len(kw_only))} ---")
        sample = kw_only.sample(min(sample_n, len(kw_only)), random_state=42)
        for _, row in sample.iterrows():
            print(f"  BENEFIT_NAME: {row['BENEFIT_NAME']}")
            print(f"  BENEFIT_SUBCATEGORY: {row['BENEFIT_SUBCATEGORY_NAME']}")
            if kw_col == "REMOTE_KW":
                print(f"  REMOTE_TYPE_NAME: {row['REMOTE_TYPE_NAME']}")
            body = str(row.get("BODY", ""))[:200]
            if body:
                print(f"  BODY snippet: {body}...")
            print()
    else:
        print(f"\n--- No Keyword+ / Structured- cases ---")

    if len(st_only) > 0:
        print(f"--- Structured+ / Keyword- ({len(st_only):,} rows) — sample {min(sample_n, len(st_only))} ---")
        sample = st_only.sample(min(sample_n, len(st_only)), random_state=42)
        for _, row in sample.iterrows():
            print(f"  BENEFIT_NAME: {row['BENEFIT_NAME']}")
            print(f"  BENEFIT_SUBCATEGORY: {row['BENEFIT_SUBCATEGORY_NAME']}")
            if kw_col == "REMOTE_KW":
                print(f"  REMOTE_TYPE_NAME: {row['REMOTE_TYPE_NAME']}")
            body = str(row.get("BODY", ""))[:200]
            if body:
                print(f"  BODY snippet: {body}...")
            print()
    else:
        print(f"--- No Structured+ / Keyword- cases ---")


Health & Wellbeing

--- Keyword+ / Structured- (2,067 rows) — sample 5 ---
  BENEFIT_NAME: ['Paid Time Off (PTO)', '401(k) Plans', 'Health Insurance', 'Dental Insurance', 'Vision Insurance']
  BENEFIT_SUBCATEGORY: ['Paid Time Off (PTO)', '401(k) Plans', 'Health Insurance', 'Dental Insurance', 'Vision Insurance']
  BODY snippet: Receptionist

Impress Communications

Calabasas, CA 91302

Urgently hiring

Job details

Job Type

Full-time

Number of hires for this role

1

Qualifications

*
*

High school or equivalent (Preferre...

  BENEFIT_NAME: ['Paid Time Off (PTO)', '401(k) Plans', 'Flexible Work Schedules', 'Discounts/Reimbursements']
  BENEFIT_SUBCATEGORY: ['Paid Time Off (PTO)', '401(k) Plans', 'Flexible Work Schedules', 'Discounts/Reimbursements']
  BODY snippet: Amazon Warehouse Attendant - Morning Shifts Available

Amazon

* Spanaway, WA
* Permanent
* Full-time

* 4 hours ago
New hires who show proof of their Covid-19 vaccination earn a $100 bonus their firs...

  BENEFIT_NAME

## 6. Coverage comparison: structured field completeness

The structured benefit fields may be sparsely populated (many rows have `[]`). Compare what fraction of postings have *any* structured benefit vs. *any* keyword-detected benefit, broken down by year.

In [36]:
data["has_structured"] = data["BENEFIT_NAME"].apply(len).gt(0)
data["has_keyword"] = data[kw_labels].any(axis=1)

coverage = data.groupby("YEAR").agg(
    n=("ID", "size"),
    pct_structured=("has_structured", "mean"),
    pct_keyword=("has_keyword", "mean"),
).round(4)
coverage["pct_structured"] = (coverage["pct_structured"] * 100).round(1)
coverage["pct_keyword"] = (coverage["pct_keyword"] * 100).round(1)
display(coverage)

,n,pct_structured,pct_keyword
YEAR,,,
2018,10199,29.4,21.2
2019,10789,34.5,25.6
2020,10615,42.3,33.0
2021,13999,48.4,40.2
2022,15529,51.7,45.7
2023,13274,56.7,50.3
2024,12304,61.7,55.2
2025,13151,62.5,53.8


## 7. Prevalence comparison by AI ROLE

Do the two approaches tell the same story about AI vs non-AI differences?

In [37]:
rows = []
for kw_col, (struct_col, name) in all_pairs.items():
    for label, source in [(kw_col, "Keyword"), (struct_col, "Structured")]:
        ai = data[data["AI ROLE"]][label].mean()
        non_ai = data[~data["AI ROLE"]][label].mean()
        rows.append({
            "Benefit": name,
            "Source": source,
            "AI (%)": round(ai * 100, 2),
            "Non-AI (%)": round(non_ai * 100, 2),
            "Diff (pp)": round((ai - non_ai) * 100, 2),
        })

comparison = pd.DataFrame(rows)
display(comparison.pivot_table(index="Benefit", columns="Source", values=["AI (%)", "Non-AI (%)", "Diff (pp)"]))

AI (%)            Diff (pp)            Non-AI (%)  \
Source             Keyword Structured   Keyword Structured    Keyword   
Benefit                                                                 
Health & Wellbeing    5.95       7.51      1.01      -0.73       4.94   
Paid Leave           27.23      26.26     -1.04      -1.06      28.27   
Parental Leave       10.42      10.12      5.85       5.70       4.56   
Remote Work          18.45      20.24     12.25      13.94       6.20   
Tuition Assistance    9.90       9.23      0.62       0.77       9.28   

                               
Source             Structured  
Benefit                        
Health & Wellbeing       8.24  
Paid Leave              27.33  
Parental Leave           4.42  
Remote Work              6.29  
Tuition Assistance       8.46